# El péndulo con `python-control`: modelo, trimado, linealización y simulación

En este cuaderno vamos a estudiar un **péndulo** al que podemos aplicar un **par** (un "giro") en su eje,
usando la librería **python-control**. Los pasos son los mismos que se siguen con cualquier sistema
dinámico, por complicado que sea (un avión, un robot, un motor...):

1. Escribir las **ecuaciones diferenciales** (EDO) del sistema.
2. Construir el **sistema** en la librería de control.
3. Calcular un punto de equilibrio (**trimado**): el péndulo quieto a **45°**.
4. **Linealizar** el sistema alrededor de ese punto.
5. **Simular** qué pasa si le damos un pequeño "empujón" de par.
6. **Comparar** la simulación lineal con la no lineal.
7. Ver una **animación** del péndulo.

```
            eje (pivote)
               O ──── aquí aplicamos el par τ
               |\
               | \
               |θ \   varilla de longitud L
               |   \
               |    ●  bola de masa m
            vertical
```

El ángulo **θ** se mide desde la vertical hacia abajo y es positivo en sentido antihorario.

**Cómo usar el cuaderno:** ejecuta las celdas **en orden**, de arriba abajo, con `Shift + Enter`.
Las celdas marcadas como *Parámetros* se pueden modificar para experimentar.

## 0. Importar las librerías

Una **librería** es un conjunto de funciones que otras personas ya han programado.
La importamos con un nombre corto (por ejemplo `np`) y luego usamos sus funciones así: `np.sin(...)`.

In [ ]:
import numpy as np                 # cálculo numérico: vectores, senos, cosenos...
import matplotlib.pyplot as plt    # para dibujar gráficas
import control as ct               # la librería de control: python-control

# Para la animación del final
from matplotlib.animation import FuncAnimation   # crea animaciones fotograma a fotograma
from IPython.display import HTML                 # muestra la animación dentro del cuaderno

print("Versión de python-control:", ct.__version__)

## 1. Las ecuaciones del sistema

Aplicando la 2ª ley de Newton para rotaciones (momento de inercia × aceleración angular = suma de pares):

$$
m L^2 \, \ddot\theta \;=\; \tau \;-\; m g L \sin\theta \;-\; b\,\dot\theta
$$

| Término | Significado |
|---|---|
| $m L^2$ | momento de inercia de la bola respecto al eje |
| $\tau$ | par que aplicamos en el eje (la **entrada** del sistema) |
| $-m g L \sin\theta$ | par del peso, que intenta devolver el péndulo hacia abajo |
| $-b\,\dot\theta$ | rozamiento en el eje, que frena el movimiento |

Es una ecuación de **segundo orden** (aparece $\ddot\theta$). La librería trabaja con ecuaciones de
**primer orden**, así que definimos dos **estados**: el ángulo $\theta$ y la velocidad angular
$\omega = \dot\theta$. Queda:

$$
\dot\theta = \omega
\qquad\qquad
\dot\omega = \frac{\tau - m g L \sin\theta - b\,\omega}{m L^2}
$$

- **Estado:** $x = [\theta,\ \omega]$
- **Entrada:** $u = [\tau]$
- **Salida** (lo que medimos): $y = [\theta]$

### Parámetros físicos

In [ ]:
# Parámetros físicos del péndulo
m = 1.0     # masa de la bola [kg]
L = 1.0     # longitud de la varilla [m]
g = 9.81    # aceleración de la gravedad [m/s^2]
b = 0.5     # coeficiente de rozamiento en el eje [N·m·s/rad]

# Los guardamos en un "diccionario": una lista de valores, cada uno con su nombre.
# Así se los podremos pasar a la librería de control de una sola vez.
parametros = {'m': m, 'L': L, 'g': g, 'b': b}

print(parametros)
print("Para leer un valor se usa su nombre entre corchetes, por ejemplo la masa:", parametros['m'])

### Las ecuaciones como funciones de Python

python-control necesita dos funciones con estos argumentos **en este orden**: `(t, x, u, params)`

- `t`: el tiempo (aquí no lo usamos, pero hay que ponerlo),
- `x`: el vector de estado $[\theta, \omega]$,
- `u`: el vector de entradas $[\tau]$,
- `params`: el diccionario de parámetros.

La primera devuelve las **derivadas** de los estados; la segunda, las **salidas**.

In [ ]:
def ecuaciones_pendulo(t, x, u, params):
    """Devuelve las derivadas del estado [dtheta/dt, domega/dt]."""

    # 1) Sacamos los parámetros del diccionario
    m = params['m']
    L = params['L']
    g = params['g']
    b = params['b']

    # 2) Separamos el vector de estado en sus dos componentes
    #    (en Python los índices empiezan en 0)
    theta = x[0]     # ángulo [rad]
    omega = x[1]     # velocidad angular [rad/s]

    # 3) La entrada: el par aplicado en el eje
    tau = u[0]       # [N·m]

    # 4) Las ecuaciones diferenciales
    dtheta = omega
    domega = (tau - m * g * L * np.sin(theta) - b * omega) / (m * L**2)

    # 5) Devolvemos las derivadas en el mismo orden que los estados
    return [dtheta, domega]


def salida_pendulo(t, x, u, params):
    """Devuelve la salida del sistema: el ángulo theta."""
    theta = x[0]
    return [theta]

**Probamos la función** antes de seguir. Si ponemos el péndulo quieto a 45° y **sin par**, el peso
debería hacer que empiece a caer, es decir, $\dot\omega$ debería salir **negativa**.

Ojo: las funciones trigonométricas de numpy trabajan en **radianes**. Para pasar de grados a radianes
usamos `np.radians(...)`, y para volver, `np.degrees(...)`.

In [ ]:
x_prueba = [np.radians(45), 0.0]   # theta = 45 grados, omega = 0
u_prueba = [0.0]                   # sin par

derivadas = ecuaciones_pendulo(0, x_prueba, u_prueba, parametros)
print("dtheta/dt =", derivadas[0], "rad/s")
print("domega/dt =", derivadas[1], "rad/s^2   <- negativa: el péndulo empieza a caer")

## 2. Construir el sistema en la librería de control

Con `ct.nlsys` ("*nonlinear system*") juntamos en un solo objeto las dos funciones, los nombres de los
estados, entradas y salidas, y los parámetros. A partir de ahora la variable `pendulo` **es** el sistema,
y se lo podemos pasar a cualquier función de la librería (trimar, linealizar, simular...).

In [ ]:
pendulo = ct.nlsys(
    ecuaciones_pendulo,          # función con las derivadas
    salida_pendulo,              # función con las salidas
    name='pendulo',              # nombre del sistema
    states=['theta', 'omega'],   # nombres de los estados (en el mismo orden que x)
    inputs=['tau'],              # nombre de la entrada
    outputs=['theta'],           # nombre de la salida
    params=parametros)           # parámetros físicos

print(pendulo)

## 3. Trimado: el péndulo en equilibrio a 45°

Sin par, el péndulo solo puede estar quieto colgando hacia abajo ($\theta = 0$). Para mantenerlo
**quieto a 45°** hay que aplicar un par constante $\tau_0$ que compense el peso.

En el equilibrio **nada cambia**: todas las derivadas son cero.

- $\dot\theta = 0 \;\Rightarrow\; \omega_0 = 0$ (está quieto)
- $\dot\omega = 0 \;\Rightarrow\; \tau_0 = m g L \sin\theta_0$ (el par compensa exactamente el peso)

Este caso es tan sencillo que lo resolvemos a mano, pero en sistemas complicados no se puede. Para eso
está `ct.find_operating_point`, que busca numéricamente el punto de equilibrio. Le decimos:

- una **estimación inicial** del estado y de la entrada,
- qué **salida** queremos fijar y con qué valor: $\theta = 45°$,
- que la **entrada** $\tau$ es **libre** (la incógnita): `input_indices=[]` significa "no fijes ninguna entrada".

Por defecto la función exige que **todas las derivadas sean cero**, que es justo la condición de equilibrio.

In [ ]:
theta_trim = np.radians(45)    # ángulo de equilibrio que queremos [rad]

punto = ct.find_operating_point(
    pendulo,
    initial_state=[theta_trim, 0.0],   # estimación inicial del estado [theta, omega]
    inputs=[0.0],                      # estimación inicial del par (luego la ajusta)
    outputs=[theta_trim],              # valor deseado de la salida: theta = 45 grados
    output_indices=[0],                # fijamos la salida número 0 (theta)
    input_indices=[])                  # no fijamos ninguna entrada: tau es la incógnita

x_trim = punto.states      # estado de equilibrio [theta, omega]
u_trim = punto.inputs      # entrada de equilibrio [tau]
tau_trim = u_trim[0]       # el par de equilibrio, como número

print("Estado de equilibrio:")
print(f"  theta = {np.degrees(x_trim[0]):.2f} grados")
print(f"  omega = {x_trim[1]:.2e} rad/s")
print(f"Par de equilibrio:  tau = {tau_trim:.4f} N·m")

# Comprobación con la fórmula obtenida a mano
tau_a_mano = m * g * L * np.sin(theta_trim)
print(f"Par calculado a mano: m*g*L*sin(45º) = {tau_a_mano:.4f} N·m")

# Comprobación de que es un equilibrio: las derivadas deben ser (casi) cero
print("Derivadas en el equilibrio:", np.array(ecuaciones_pendulo(0, x_trim, u_trim, parametros)))

## 4. Linealización

Las ecuaciones del péndulo son **no lineales** (aparece $\sin\theta$). Muchas herramientas de control
(funciones de transferencia, polos, Bode...) solo sirven para sistemas **lineales**. Por eso
**linealizamos**: aproximamos el sistema por uno lineal que es válido **cerca del punto de equilibrio**.

Trabajamos con **desviaciones** respecto al equilibrio:

$$
\Delta\theta = \theta - \theta_0, \qquad \Delta\omega = \omega - 0, \qquad \Delta\tau = \tau - \tau_0
$$

y el modelo lineal es

$$
\Delta\dot x = A\,\Delta x + B\,\Delta u, \qquad \Delta y = C\,\Delta x + D\,\Delta u
$$

Haciendo las derivadas a mano ($\sin\theta \approx \sin\theta_0 + \cos\theta_0\,\Delta\theta$):

$$
A = \begin{bmatrix} 0 & 1 \\[2pt] -\dfrac{g}{L}\cos\theta_0 & -\dfrac{b}{m L^2} \end{bmatrix},
\qquad
B = \begin{bmatrix} 0 \\[2pt] \dfrac{1}{m L^2} \end{bmatrix},
\qquad
C = \begin{bmatrix} 1 & 0 \end{bmatrix},
\qquad
D = 0
$$

`ct.linearize` calcula estas matrices automáticamente (numéricamente).

In [ ]:
lin = ct.linearize(pendulo, x_trim, u_trim, name='pendulo_lineal')

# np.round(..., 4) redondea a 4 decimales para que se lea mejor
print("Matriz A (calculada por la librería):")
print(np.round(lin.A, 4))
print("Matriz B:")
print(np.round(lin.B, 4))
print("Matriz C:", lin.C, "   Matriz D:", lin.D)

# Comparamos con las fórmulas hechas a mano
A_a_mano = np.array([[0.0,                           1.0],
                     [-g / L * np.cos(theta_trim),   -b / (m * L**2)]])
print("\nMatriz A (a mano):")
print(np.round(A_a_mano, 4))

### Polos, frecuencia natural y amortiguamiento

Los **polos** del sistema lineal (los autovalores de $A$) nos dicen cómo se mueve el péndulo
cerca del equilibrio:

- Si tienen **parte real negativa**, el equilibrio es **estable**: el péndulo vuelve a él.
- Si son **complejos**, el péndulo **oscila**. La frecuencia natural $\omega_n$ indica lo rápido que oscila
  y el amortiguamiento $\zeta$ lo rápido que se frenan las oscilaciones ($\zeta$ pequeño = oscila mucho).

`ct.damp` muestra todo esto en una tabla.

In [ ]:
polos = lin.poles()
print("Polos:", np.round(polos, 4))
print()
ct.damp(lin);   # tabla con amortiguamiento y frecuencia natural de cada polo

In [ ]:
# Función de transferencia: Delta_theta(s) / Delta_tau(s)
G = ct.tf(lin)
print("Función de transferencia G(s) =", G)

# Ganancia estática: cuánto se desvía theta (al final) por cada N·m de par extra
ganancia = ct.dcgain(G)
print(f"Ganancia estática: {ganancia:.4f} rad por N·m  =  {np.degrees(ganancia):.2f} grados por N·m")

# Periodo de la oscilación
wn = abs(polos[0])                     # frecuencia natural [rad/s]
zeta = -polos[0].real / wn             # amortiguamiento [-]
periodo = 2 * np.pi / abs(polos[0].imag)
print(f"wn = {wn:.3f} rad/s,  zeta = {zeta:.3f},  periodo de oscilación = {periodo:.2f} s")

## 5. Simulación no lineal ante una perturbación de par

Partimos del péndulo **en equilibrio a 45°** y, durante un rato, le sumamos un pequeño par extra
$\Delta\tau$ (un "empujón"). Después el par vuelve a su valor de equilibrio $\tau_0$.

```
  par τ
    ^        ┌──────────┐  ← τ0 + Δτ
    |        |          |
    |────────┘          └──────────────  ← τ0
    +────────┬──────────┬──────────────> tiempo
          t_inicio   t_inicio + duración
```

Simulamos con `ct.input_output_response(sistema, T, U, X0)`:

- `T`: vector de tiempos,
- `U`: valor de la entrada en cada instante de `T`,
- `X0`: estado inicial.

### Parámetros (modifícalos aquí)

In [ ]:
t_final  = 20.0    # duración de la simulación [s]
dt       = 0.01    # intervalo entre instantes guardados [s]

dtau     = 0.5     # tamaño del empujón de par [N·m]
t_inicio = 1.0     # instante en que empieza el empujón [s]
duracion = 2.0     # cuánto dura el empujón [s]

In [ ]:
# Vector de tiempos: 0, 0.01, 0.02, ..., hasta t_final
T = np.arange(0, t_final, dt)

# "pulso" vale True (=1) durante el empujón y False (=0) el resto del tiempo
pulso = (T >= t_inicio) & (T < t_inicio + duracion)

# Par aplicado en cada instante: el de equilibrio más el empujón cuando toca
tau = tau_trim + dtau * pulso

# Simulación del sistema NO LINEAL, empezando en el equilibrio
# (max_step limita el paso del integrador para que no "se salte" el inicio del pulso)
resp_nl = ct.input_output_response(pendulo, T, tau, X0=x_trim,
                                   solve_ivp_kwargs={'max_step': dt})

# resp_nl.states tiene una fila por estado: fila 0 = theta, fila 1 = omega
theta_nl = resp_nl.states[0]
omega_nl = resp_nl.states[1]

# --- Gráficas: par, ángulo y velocidad angular ---
fig, ejes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

ejes[0].plot(T, tau, 'k')
ejes[0].set_ylabel('par τ [N·m]')
ejes[0].set_title(f'Respuesta no lineal a un empujón de Δτ = {dtau} N·m durante {duracion} s')

ejes[1].plot(T, np.degrees(theta_nl), 'b', label='simulación')
ejes[1].axhline(np.degrees(theta_trim), color='r', linestyle='--', label='equilibrio (45°)')
ejes[1].set_ylabel('θ [grados]')
ejes[1].legend()

ejes[2].plot(T, np.degrees(omega_nl), 'b')
ejes[2].set_ylabel('ω [grados/s]')
ejes[2].set_xlabel('tiempo [s]')

for eje in ejes:
    eje.grid(True)
plt.tight_layout()
plt.show()

**Qué se ve:** al empezar el empujón el péndulo sube y oscila alrededor de un ángulo algo mayor que 45°.
Al terminar el empujón vuelve a oscilar, ahora alrededor de 45°, y las oscilaciones se van frenando
por el rozamiento. El periodo de las oscilaciones coincide con el que calculamos en la sección 4.

## 6. Comparación: modelo lineal frente a no lineal

Ahora simulamos el **modelo lineal** con el mismo empujón y comparamos. Hay que tener cuidado:
el modelo lineal trabaja con **desviaciones**, así que

1. su entrada es $\Delta\tau = \tau - \tau_0$ (solo el empujón),
2. su estado inicial es $\Delta x = [0, 0]$ (empezamos justo en el equilibrio),
3. su salida es $\Delta\theta$, y para obtener el ángulo real hay que sumarle $\theta_0$.

Para sistemas lineales se usa `ct.forced_response`, que es más rápida que `input_output_response`.

In [ ]:
# Simulación del modelo LINEAL (en desviaciones)
resp_lin = ct.forced_response(lin, T, tau - tau_trim, X0=[0.0, 0.0])

# Ángulo real = equilibrio + desviación
theta_lin = theta_trim + resp_lin.outputs

# --- Gráfica comparativa ---
fig, ejes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ejes[0].plot(T, np.degrees(theta_nl), 'b', linewidth=2.5, label='no lineal')
ejes[0].plot(T, np.degrees(theta_lin), 'r--', linewidth=2, label='lineal')
ejes[0].set_ylabel('θ [grados]')
ejes[0].set_title(f'Empujón de Δτ = {dtau} N·m')
ejes[0].legend()

ejes[1].plot(T, np.degrees(theta_nl - theta_lin), 'k')
ejes[1].set_ylabel('diferencia [grados]')
ejes[1].set_xlabel('tiempo [s]')

for eje in ejes:
    eje.grid(True)
plt.tight_layout()
plt.show()

print(f"Máxima diferencia entre los dos modelos: {np.max(np.abs(np.degrees(theta_nl - theta_lin))):.3f} grados")

Con un empujón pequeño los dos modelos **casi coinciden**: la linealización es una buena aproximación.

### ¿Y con un empujón grande?

La linealización solo es válida **cerca** del equilibrio. Para verlo, repetimos la comparación con
varios tamaños de empujón. Como vamos a hacer lo mismo varias veces, metemos los pasos anteriores en una
**función** `simular(dtau)`, que devuelve el par aplicado y el ángulo según los dos modelos.

In [ ]:
def simular(dtau):
    """Simula el empujón de tamaño dtau con los dos modelos.
    Devuelve: par aplicado, theta no lineal y theta lineal (en radianes)."""
    tau = tau_trim + dtau * pulso
    resp_nl  = ct.input_output_response(pendulo, T, tau, X0=x_trim,
                                        solve_ivp_kwargs={'max_step': dt})
    resp_lin = ct.forced_response(lin, T, tau - tau_trim, X0=[0.0, 0.0])
    theta_nl  = resp_nl.states[0]
    theta_lin = theta_trim + resp_lin.outputs
    return tau, theta_nl, theta_lin


# Probamos tres tamaños de empujón (puedes cambiarlos)
empujones = [0.5, 1.5, 2.5]     # [N·m]

fig, ejes = plt.subplots(1, len(empujones), figsize=(15, 4), sharey=True)
for i in range(len(empujones)):           # i = 0, 1, 2
    tau_i, theta_nl_i, theta_lin_i = simular(empujones[i])
    ejes[i].plot(T, np.degrees(theta_nl_i), 'b', linewidth=2.5, label='no lineal')
    ejes[i].plot(T, np.degrees(theta_lin_i), 'r--', linewidth=2, label='lineal')
    ejes[i].axhline(45, color='gray', linestyle=':')
    ejes[i].set_title(f'Δτ = {empujones[i]} N·m')
    ejes[i].set_xlabel('tiempo [s]')
    ejes[i].grid(True)
ejes[0].set_ylabel('θ [grados]')
ejes[0].legend()
plt.tight_layout()
plt.show()

**Conclusión:** cuanto mayor es el empujón, más se aleja el péndulo de 45° y más se separan los dos
modelos. Con $\Delta\tau = 2.5$ N·m el péndulo real sube hasta unos 110°, mientras que el modelo lineal
predice solo unos 80°: se equivoca tanto en la amplitud como en el periodo de las oscilaciones, porque
$\sin\theta$ ya no se parece a la recta tangente en 45°.

El modelo lineal es muy útil para **diseñar** controladores, pero hay que **comprobar** siempre el
resultado con el modelo no lineal.

## 7. Animación del péndulo

Para dibujar el péndulo necesitamos la posición de la bola. Con el pivote en el origen y $\theta$
medido desde la vertical hacia abajo:

$$
x_{bola} = L \sin\theta, \qquad y_{bola} = -L\cos\theta
$$

Una animación es una sucesión de **fotogramas**. `FuncAnimation` llama a una función nuestra
(`dibujar_fotograma`) una vez por fotograma; esa función solo tiene que **mover** la varilla y la bola
a su posición en ese instante.

Dibujamos a la vez el péndulo **no lineal** (azul) y el **lineal** (naranja, semitransparente), para ver
la diferencia. El texto se pone en rojo mientras dura el empujón.

### Parámetros (modifícalos aquí)

In [ ]:
dtau_animacion = 2.5   # empujón que vamos a animar [N·m] (con 0.5 apenas se ve el movimiento)
paso = 8               # dibujamos 1 de cada 'paso' instantes (8 x 0.01 s = un fotograma cada 0.08 s)

In [ ]:
# 1) Simulamos y calculamos la posición de la bola en cada instante
tau_a, theta_nl_a, theta_lin_a = simular(dtau_animacion)

x_nl  =  L * np.sin(theta_nl_a)      # posición de la bola (modelo no lineal)
y_nl  = -L * np.cos(theta_nl_a)
x_lin =  L * np.sin(theta_lin_a)     # posición de la bola (modelo lineal)
y_lin = -L * np.cos(theta_lin_a)

# 2) Preparamos la figura (lo que no se mueve se dibuja una sola vez)
fig, ax = plt.subplots(figsize=(5, 5), dpi=72)   # dpi bajo = animación más ligera
ax.set_xlim(-1.3 * L, 1.3 * L)
ax.set_ylim(-1.3 * L, 1.3 * L)
ax.set_aspect('equal')               # misma escala en x e y, para que los ángulos se vean bien
ax.grid(True)
ax.set_title(f'Péndulo: empujón de Δτ = {dtau_animacion} N·m')

ax.plot([0, 0], [0, -1.2 * L], color='gray', linestyle=':')     # vertical
ax.plot([0, 1.2 * L * np.sin(theta_trim)], [0, -1.2 * L * np.cos(theta_trim)],
        color='gray', linestyle='--', label='equilibrio (45°)')   # línea de 45 grados
ax.plot(0, 0, 'ks', markersize=10)                              # el pivote

# 3) Creamos vacíos los elementos que se van a mover
#    'o-' = línea con círculos; markevery=[1] pone el círculo solo en el 2º punto (la bola)
pendulo_nl,  = ax.plot([], [], 'o-', color='tab:blue', linewidth=3, markersize=18,
                       markevery=[1], label='no lineal')
pendulo_lin, = ax.plot([], [], 'o-', color='tab:orange', linewidth=3, markersize=18,
                       markevery=[1], alpha=0.5, label='lineal')
texto = ax.text(-1.25 * L, 1.15 * L, '', va='top', family='monospace')
ax.legend(loc='upper right', fontsize=8, markerscale=0.4)


# 4) Función que dibuja el fotograma correspondiente al instante número i
def dibujar_fotograma(i):
    # Varilla: línea desde el pivote (0, 0) hasta la bola
    pendulo_nl.set_data([0, x_nl[i]], [0, y_nl[i]])
    pendulo_lin.set_data([0, x_lin[i]], [0, y_lin[i]])

    # Texto con los valores del instante actual
    texto.set_text(f't = {T[i]:5.2f} s\n'
                   f'θ = {np.degrees(theta_nl_a[i]):5.1f}°\n'
                   f'τ = {tau_a[i]:5.2f} N·m')
    if pulso[i]:
        texto.set_color('red')       # durante el empujón
    else:
        texto.set_color('black')

    # Devolvemos los elementos que han cambiado
    return pendulo_nl, pendulo_lin, texto


# 5) Creamos la animación: un fotograma por cada 'paso' instantes
indices = range(0, len(T), paso)          # 0, 8, 16, 24, ...
anim = FuncAnimation(fig, dibujar_fotograma, frames=indices,
                     interval=1000 * dt * paso,   # ms entre fotogramas -> velocidad real
                     blit=True)
plt.close(fig)        # para que no aparezca además una imagen fija

# 6) La mostramos en el cuaderno, con botones de reproducción
plt.rcParams['animation.embed_limit'] = 60    # permite animaciones de hasta 60 MB
HTML(anim.to_jshtml())

Para **guardar** la animación como GIF (por ejemplo, para una presentación), ejecuta en una celda nueva:

```python
anim.save('pendulo.gif', writer='pillow', fps=20)
```

## 8. Para practicar

1. Cambia `dtau` en la sección 5 y busca a partir de qué tamaño de empujón la diferencia entre el modelo
   lineal y el no lineal supera 1 grado.
2. Pon `b = 0` (sin rozamiento) en los parámetros físicos y vuelve a ejecutar todo. ¿Qué les pasa a los
   polos? ¿Se frenan las oscilaciones?
3. Trima el péndulo a **90°** y luego a **150°**. ¿Cómo cambian el par de equilibrio, la frecuencia
   natural y el periodo? ¿Qué pasaría a 180° (péndulo invertido)? Mira el signo de $\cos\theta_0$ en la
   matriz $A$.
4. Cambia el empujón por un **escalón** (un par extra que no se quita nunca) haciendo
   `duracion = t_final`. ¿Dónde se queda el péndulo al final? Compáralo con la ganancia estática de la
   sección 4.
5. Prueba un empujón de **3 N·m** en la sección 6 o en la animación. ¿Qué hace el péndulo no lineal?
   Pista: compara el par total $\tau_0 + \Delta\tau$ con el par máximo que puede hacer el peso,
   $m g L$. ¿Lo predice el modelo lineal?